# Nemotron Reasoning Challenge — Solver-Distilled LoRA

A self-contained solution for the NVIDIA Nemotron Model Reasoning Challenge.

**Approach.** The hidden rule behind each task (bit operations, gravity, unit
conversion, numeral systems, letter ciphers) is recovered by deterministic
solvers and distilled into a verified chain-of-thought curriculum — every
training example carries a correct reasoning trace ending in `\boxed{answer}`.
A rank-32 LoRA adapter is then fine-tuned on Nemotron-3-Nano-30B-A3B.

**Pipeline.** Build the curriculum from the competition `train.csv` → fine-tune
the LoRA adapter (BF16) → package `/kaggle/working/submission.zip`.

**Run settings.** RTX 6000 accelerator, Internet disabled. Attach the
competition data, the `metric/nemotron-3-nano-30b-a3b-bf16` model, and the
NVIDIA utility source listed in `kernel-metadata.json`.


In [ ]:
import importlib.util
import site
from pathlib import Path

cutlass_candidates = [
    Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages'),
    Path('/kaggle/usr/lib/nvidia-utility-script/nvidia_cutlass_dsl/python_packages'),
]
cutlass_candidates += sorted(Path('/kaggle/usr/lib').glob('**/nvidia_cutlass_dsl/python_packages'))

added_cutlass = []
for path in cutlass_candidates:
    if path.exists() and str(path) not in added_cutlass:
        site.addsitedir(str(path))
        added_cutlass.append(str(path))
print('added CUTLASS paths:', added_cutlass)

required = ['peft', 'mamba_ssm', 'causal_conv1d', 'cutlass']
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    print('debug /kaggle/usr/lib:', [str(p) for p in Path('/kaggle/usr/lib').glob('*')][:50])
    raise RuntimeError(
        'Missing required offline packages: ' + ', '.join(missing) + '. '
        'Attach ryanholbrook/nvidia-utility-script as a kernel source, then rerun.'
    )


In [ ]:
# Blackwell ptxas fix: chmod/copy the read-only binary and monkeypatch get_ptxas().
import os
import shutil
import stat
from pathlib import Path

_EXEC = stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH
_dst = Path('/kaggle/working/triton-bin')
_dst.mkdir(parents=True, exist_ok=True)

def _make_exec(p):
    # Copy to writable dir on EROFS/EACCES; both are OSError subclasses.
    try:
        p.chmod(p.stat().st_mode | _EXEC)
        if os.access(str(p), os.X_OK):
            return str(p)
    except OSError:
        pass
    d = _dst / p.name
    shutil.copy2(p, d)
    d.chmod(d.stat().st_mode | _EXEC)
    return str(d)

_ptxas = {}
for _p in sorted(Path('/kaggle/usr/lib').glob('**/triton/backends/nvidia/bin/ptxas*')):
    _ptxas[_p.name] = _make_exec(_p)

assert _ptxas, (
    'No ptxas binary found under /kaggle/usr/lib/**/triton/backends/nvidia/bin. '
    'Attach ryanholbrook/nvidia-utility-script as a kernel source.'
)
print('ptxas executables ready:', _ptxas)

_generic = _ptxas.get('ptxas') or next(iter(_ptxas.values()))
_blackwell = _ptxas.get('ptxas-blackwell', _generic)
os.environ.setdefault('TRITON_PTXAS_PATH', _generic)

# Monkeypatch both generic and Blackwell branches to the executable copies.
from triton.backends.nvidia import compiler as _nvcomp
try:
    from triton.knobs import NvidiaTool
    _make_tool = NvidiaTool.from_path
except Exception:
    class _ToolShim:
        def __init__(self, path):
            self.path = path
    _make_tool = _ToolShim

def _get_ptxas(arch):
    return _make_tool(_blackwell if arch >= 100 else _generic)

_nvcomp.get_ptxas = _get_ptxas
print('patched triton get_ptxas ->', {'>=100': _blackwell, '<100': _generic})


In [ ]:
from pathlib import Path
import os

WORK = Path("/kaggle/working/nemotron_challenge")
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print("working directory:", WORK)


## Pipeline modules

The cells below materialize the curriculum-building modules into the working
directory so they import and log like ordinary files.


In [ ]:
from pathlib import Path
WORK = Path('/kaggle/working/nemotron_challenge')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'solvers.py').write_text('"""\nDeterministic code-based solvers for each problem type.\nThese bypass the LLM entirely for problems that have exact algebraic/combinatorial solutions.\nThe LLM is only used as fallback when code can\'t find a consistent rule.\n"""\n\nimport re\nimport itertools\nimport statistics\nfrom typing import Optional\n\n\ndef _parse_examples(prompt: str) -> tuple[list[str], list[str], str]:\n    """Extract (inputs, outputs, query) from any problem prompt."""\n    lines = prompt.strip().splitlines()\n    inputs, outputs = [], []\n    query = ""\n    for line in lines:\n        if "->" in line:\n            parts = line.split("->", 1)\n            lhs = parts[0].strip()\n            rhs = parts[1].strip()\n            # skip header/description lines like "Here are some examples of input -> output:"\n            # valid data lines: both sides are short tokens (numbers, binary, words)\n            # reject if lhs contains more than ~6 words (it\'s a description)\n            if len(lhs.split()) <= 5 and len(rhs.split()) <= 10:\n                inputs.append(lhs)\n                outputs.append(rhs)\n        elif line.strip().lower().startswith("now") or "determine" in line.lower() or "convert" in line.lower() or "write" in line.lower() or "decrypt" in line.lower():\n            m = re.search(r"(?:for:|following:?|number\\s+|text:\\s*)(.+)$", line, re.IGNORECASE)\n            if m:\n                query = m.group(1).strip().rstrip(".")\n    # fallback: last non-arrow, non-header line\n    if not query:\n        for line in reversed(lines):\n            line = line.strip()\n            if line and "->" not in line and not line.lower().startswith("in alice") and not line.lower().startswith("here") and not line.lower().startswith("now"):\n                query = line\n                break\n    return inputs, outputs, query\n\n\n# ── Bit manipulation ────────────────────────────────────────────────────────\n\ndef _apply_bit_op(val: int, op: str, param: int = 0) -> int:\n    if op == "NOT":  return (~val) & 0xFF\n    if op == "REV":  return int(f"{val:08b}"[::-1], 2)\n    if op == "ROL":  return ((val << param) | (val >> (8 - param))) & 0xFF\n    if op == "ROR":  return ((val >> param) | (val << (8 - param))) & 0xFF\n    if op == "XOR":  return val ^ param\n    if op == "AND":  return val & param\n    if op == "OR":   return val | param\n    if op == "SHL":  return (val << param) & 0xFF\n    if op == "SHR":  return (val >> param) & 0xFF\n    return val\n\ndef _try_single_ops(examples: list[tuple[int, int]]) -> Optional[tuple]:\n    for op in ("NOT", "REV"):\n        if all(_apply_bit_op(i, op) == o for i, o in examples):\n            return (op, 0)\n    for op in ("ROL", "ROR", "SHL", "SHR"):\n        for p in range(1, 8):\n            if all(_apply_bit_op(i, op, p) == o for i, o in examples):\n                return (op, p)\n    for p in range(256):\n        for op in ("XOR", "AND", "OR"):\n            if all(_apply_bit_op(i, op, p) == o for i, o in examples):\n                return (op, p)\n    return None\n\ndef _try_two_ops(examples: list[tuple[int, int]]) -> Optional[tuple]:\n    candidates = [\n        ("NOT", 0), ("REV", 0),\n        *[("ROL", p) for p in range(1, 8)],\n        *[("ROR", p) for p in range(1, 8)],\n        *[("XOR", p) for p in range(256)],\n    ]\n    for op1, p1 in candidates:\n        mid = [(_apply_bit_op(i, op1, p1), o) for i, o in examples]\n        r = _try_single_ops(mid)\n        if r:\n            return (op1, p1, r[0], r[1])\n    return None\n\ndef _learn_per_bit(examples: list[tuple[int, int]], query: int) -> Optional[str]:\n    """\n    Learn each output bit as an independent boolean function of input bits.\n    Tries (in order): constant 0/1, single bit, NOT single bit,\n    and XOR/AND/OR of every pair of input bits.\n    """\n    result = 0\n    for out_pos in range(8):\n        target = [(e[1] >> out_pos) & 1 for e in examples]\n\n        # constant\n        if all(b == 0 for b in target):\n            result |= (0 << out_pos)\n            continue\n        if all(b == 1 for b in target):\n            result |= (1 << out_pos)\n            continue\n\n        found = False\n        # single input bit or its NOT\n        for j in range(8):\n            bits     = [(e[0] >> j) & 1 for e in examples]\n            not_bits = [1 - b for b in bits]\n            if bits == target:\n                result |= (((query >> j) & 1) << out_pos)\n                found = True; break\n            if not_bits == target:\n                result |= ((1 - ((query >> j) & 1)) << out_pos)\n                found = True; break\n\n        if found:\n            continue\n\n        # two input bits combined with XOR / AND / OR / XNOR\n        for j1 in range(8):\n            for j2 in range(j1, 8):\n                b1 = [(e[0] >> j1) & 1 for e in examples]\n                b2 = [(e[0] >> j2) & 1 for e in examples]\n                combos = {\n                    "XOR":  [a ^ b      for a, b in zip(b1, b2)],\n                    "AND":  [a & b      for a, b in zip(b1, b2)],\n                    "OR":   [a | b      for a, b in zip(b1, b2)],\n                    "XNOR": [1-(a ^ b)  for a, b in zip(b1, b2)],\n                    "NAND": [1-(a & b)  for a, b in zip(b1, b2)],\n                    "NOR":  [1-(a | b)  for a, b in zip(b1, b2)],\n                }\n                for op, bits in combos.items():\n                    if bits == target:\n                        q1 = (query >> j1) & 1\n                        q2 = (query >> j2) & 1\n                        if op == "XOR":  qb = q1 ^ q2\n                        elif op == "AND": qb = q1 & q2\n                        elif op == "OR":  qb = q1 | q2\n                        elif op == "XNOR": qb = 1 - (q1 ^ q2)\n                        elif op == "NAND": qb = 1 - (q1 & q2)\n                        elif op == "NOR":  qb = 1 - (q1 | q2)\n                        result |= (qb << out_pos)\n                        found = True; break\n                if found: break\n\n        if found:\n            continue\n\n        # three input bits ANDed together (with optional NOT on each)\n        # covers patterns like: bit5 AND NOT(bit6) AND NOT(bit7)\n        for j1 in range(8):\n            for j2 in range(j1+1, 8):\n                for j3 in range(j2+1, 8):\n                    for n1, n2, n3 in itertools.product((0, 1), repeat=3):\n                        b1 = [((e[0] >> j1) & 1) ^ n1 for e in examples]\n                        b2 = [((e[0] >> j2) & 1) ^ n2 for e in examples]\n                        b3 = [((e[0] >> j3) & 1) ^ n3 for e in examples]\n                        bits = [a & b & c for a, b, c in zip(b1, b2, b3)]\n                        if bits == target:\n                            q1 = ((query >> j1) & 1) ^ n1\n                            q2 = ((query >> j2) & 1) ^ n2\n                            q3 = ((query >> j3) & 1) ^ n3\n                            result |= ((q1 & q2 & q3) << out_pos)\n                            found = True; break\n                    if found: break\n                if found: break\n\n        if not found:\n            return None  # can\'t express this bit with available ops\n\n    return f"{result:08b}"\n\ndef solve_bit(prompt: str) -> Optional[str]:\n    inputs, outputs, query = _parse_examples(prompt)\n    if not inputs or not query:\n        return None\n    try:\n        ex = [(int(i, 2), int(o, 2)) for i, o in zip(inputs, outputs)]\n        q  = int(query, 2)\n    except ValueError:\n        return None\n\n    # Try holistic ops first (fast)\n    for fn in (_try_single_ops, _try_two_ops):\n        op = fn(ex)\n        if op and len(op) == 2:\n            return f"{_apply_bit_op(q, op[0], op[1]):08b}"\n        if op and len(op) == 4:\n            mid = _apply_bit_op(q, op[0], op[1])\n            return f"{_apply_bit_op(mid, op[2], op[3]):08b}"\n\n    # Fall back to per-bit boolean function learning\n    return _learn_per_bit(ex, q)\n\n\n# ── Physics (gravity) ───────────────────────────────────────────────────────\n\ndef solve_gravity(prompt: str) -> Optional[str]:\n    # extract (t, d) pairs\n    pairs = re.findall(r"t\\s*=\\s*([\\d.]+)\\s*s.*?distance\\s*=\\s*([\\d.]+)", prompt, re.IGNORECASE)\n    query = re.search(r"t\\s*=\\s*([\\d.]+)\\s*s\\s*given", prompt, re.IGNORECASE)\n    if not pairs or not query:\n        return None\n    try:\n        tds = [(float(t), float(d)) for t, d in pairs]\n        q_t = float(query.group(1))\n        g_vals = [2 * d / (t ** 2) for t, d in tds]\n        g = statistics.median(g_vals)\n        return str(round(0.5 * g * q_t ** 2, 2))\n    except Exception:\n        return None\n\n\n# ── Unit conversion ─────────────────────────────────────────────────────────\n\ndef solve_unit(prompt: str) -> Optional[str]:\n    pairs = re.findall(r"([\\d.]+)\\s*(?:m|km|kg|s)?\\s+becomes\\s+([\\d.]+)", prompt, re.IGNORECASE)\n    query = re.search(r"convert.*?:\\s*([\\d.]+)", prompt, re.IGNORECASE)\n    if not pairs or not query:\n        return None\n    try:\n        factors = [float(b) / float(a) for a, b in pairs if float(a) != 0]\n        k = statistics.median(factors)\n        return str(round(k * float(query.group(1)), 2))\n    except Exception:\n        return None\n\n\n# ── Numeral system ──────────────────────────────────────────────────────────\n\ndef solve_numeral(prompt: str) -> Optional[str]:\n    """Build a direct lookup from examples and return if query matches a seen input."""\n    inputs, outputs, query = _parse_examples(prompt)\n    if not inputs or not query:\n        return None\n    lookup = dict(zip(inputs, outputs))\n    # direct match\n    if query in lookup:\n        return lookup[query]\n    # try to detect Roman numeral pattern\n    try:\n        num = int(query)\n        # check if outputs look like Roman numerals\n        if all(re.match(r\'^[IVXLCDM]+$\', o) for o in outputs):\n            return _to_roman(num)\n    except ValueError:\n        pass\n    return None\n\ndef _to_roman(n: int) -> str:\n    val = [1000,900,500,400,100,90,50,40,10,9,5,4,1]\n    sym = ["M","CM","D","CD","C","XC","L","XL","X","IX","V","IV","I"]\n    result = ""\n    for i, v in enumerate(val):\n        while n >= v:\n            result += sym[i]\n            n -= v\n    return result\n\n\n# ── Cipher (letter substitution) ────────────────────────────────────────────\n\ndef build_cipher_map(prompt: str) -> tuple[dict[str, str], str]:\n    """Return (char_map, query). char_map may be incomplete."""\n    example_lines = re.findall(r"([a-z ]+)\\s*->\\s*([a-z ]+)", prompt.lower())\n    char_map: dict[str, str] = {}\n    for cipher_sent, plain_sent in example_lines:\n        cw = cipher_sent.strip().split()\n        pw = plain_sent.strip().split()\n        if len(cw) != len(pw):\n            continue\n        for cword, pword in zip(cw, pw):\n            if len(cword) != len(pword):\n                continue\n            for cc, pc in zip(cword, pword):\n                char_map[cc] = pc  # last write wins on conflict\n\n    query = ""\n    for line in reversed(prompt.strip().splitlines()):\n        line = line.strip().lower()\n        if re.match(r\'^[a-z ]+$\', line) and line:\n            query = line\n            break\n    return char_map, query\n\n\ndef solve_cipher(prompt: str) -> Optional[str]:\n    """Build letter-level substitution map and decode query.\n    Returns None if any query character is unknown (caller should use LLM)."""\n    char_map, query = build_cipher_map(prompt)\n    if not query or not char_map:\n        return None\n    result = ""\n    for ch in query:\n        if ch == " ":\n            result += " "\n        elif ch in char_map:\n            result += char_map[ch]\n        else:\n            return None  # unknown char — fall back to LLM\n    return result.strip()\n\n\ndef cipher_hint(prompt: str) -> str:\n    """Return a context string with the known mappings for LLM-assisted decoding."""\n    char_map, query = build_cipher_map(prompt)\n    if not char_map:\n        return ""\n    known = ", ".join(f"{k}→{v}" for k, v in sorted(char_map.items()))\n    partial = "".join(char_map.get(c, f"[{c}?]") if c != " " else " " for c in query)\n    return (\n        f"Known letter mappings: {known}\\n"\n        f"Partial decode of query \'{query}\': {partial}\\n"\n        f"Fill in the [?] characters to complete the decoded sentence."\n    )\n\n\n# ── Symbol transform ─────────────────────────────────────────────────────────\n\ndef solve_symbol(prompt: str) -> Optional[str]:\n    """Build symbol-level substitution map (character by character or word by word)."""\n    inputs, outputs, query = _parse_examples(prompt)\n    if not inputs or not query:\n        return None\n\n    # try character-level map\n    char_map: dict[str, str] = {}\n    for inp, out in zip(inputs, outputs):\n        if len(inp) != len(out):\n            char_map = {}\n            break\n        for ci, co in zip(inp, out):\n            if ci in char_map and char_map[ci] != co:\n                char_map = {}\n                break\n            char_map[ci] = co\n\n    if char_map:\n        result = ""\n        for ch in query:\n            if ch in char_map:\n                result += char_map[ch]\n            else:\n                result = ""\n                break\n        if result:\n            return result\n\n    # try word-level map\n    word_map: dict[str, str] = {}\n    for inp, out in zip(inputs, outputs):\n        iw = inp.split()\n        ow = out.split()\n        if len(iw) != len(ow):\n            return None\n        for wi, wo in zip(iw, ow):\n            if wi in word_map and word_map[wi] != wo:\n                return None\n            word_map[wi] = wo\n\n    query_words = query.split()\n    if all(w in word_map for w in query_words):\n        return " ".join(word_map[w] for w in query_words)\n\n    return None\n\n\n# ── Dispatcher ───────────────────────────────────────────────────────────────\n\nSOLVERS = {\n    "bit":    solve_bit,\n    "gravity": solve_gravity,\n    "unit":   solve_unit,\n    "numeral": solve_numeral,\n    "cipher": solve_cipher,\n    "symbol": solve_symbol,\n}\n\ndef code_solve(problem_type: str, prompt: str) -> Optional[str]:\n    solver = SOLVERS.get(problem_type)\n    if solver is None:\n        return None\n    return solver(prompt)\n', encoding='utf-8')
print('wrote solvers.py')


In [ ]:
from pathlib import Path
WORK = Path('/kaggle/working/nemotron_challenge')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'makers.py').write_text('"""\nVerified chain-of-thought makers for the data pipeline.\n\nFor each problem type, a maker parses the prompt, derives the hidden rule, and\nreturns (cot_trace, computed_answer) — or None if the rule can\'t be recovered.\nCallers verify computed_answer against the gold label before keeping an example,\nso only correct (problem, trace, answer) triples enter the curriculum.\n\nSelf-contained: depends only on solvers.py (bit primitives + example parsing).\n"""\nimport re\nimport statistics\n\nfrom solvers import (\n    solve_bit, _apply_bit_op, _try_single_ops, _try_two_ops, _parse_examples,\n)\n\n\n# ── Type detection ──────────────────────────────────────────────────────────\ndef detect_type(prompt: str) -> str:\n    p = prompt.lower()\n    if "gravitational" in p or ("distance" in p and "t =" in p and "d = 0.5" in p):\n        return "gravity"\n    if "unit conversion" in p or " becomes " in p:\n        return "unit"\n    if "numeral system" in p:\n        return "numeral"\n    if re.search(r"\\b[01]{8}\\b", prompt):\n        return "bit"\n    if re.search(r"[^a-zA-Z\\s\\d\\->.,:;!?\'\\"()\\[\\]{}]", prompt):\n        return "symbol"\n    return "cipher"\n\n\n# ── Gravity ───────────────────────────────────────────────────────────────────\ndef make_gravity_cot(prompt: str):\n    pairs = re.findall(r"t\\s*=\\s*([\\d.]+)\\s*s.*?distance\\s*=\\s*([\\d.]+)", prompt, re.IGNORECASE)\n    q_match = re.search(r"t\\s*=\\s*([\\d.]+)\\s*s\\s*given", prompt, re.IGNORECASE)\n    if not pairs or not q_match:\n        return None\n    try:\n        tds = [(float(t), float(d)) for t, d in pairs]\n        q_t = float(q_match.group(1))\n        g_vals = [2 * d / t**2 for t, d in tds]\n        g = statistics.median(g_vals)\n        d_ans = round(0.5 * g * q_t**2, 2)\n\n        lines = ["Formula: d = 0.5·g·t²  →  g = 2d/t²"]\n        for t, d in tds:\n            g_i = round(2 * d / t**2, 4)\n            lines.append(f"  t={t}s, d={d}m  →  g = 2·{d}/{t}² = {round(2*d,4)}/{round(t**2,4)} = {g_i}")\n        lines.append(f"Median g = {round(g,4)} m/s²")\n        lines.append(f"Query t={q_t}s  →  d = 0.5·{round(g,4)}·{q_t}² = {d_ans}")\n        return "\\n".join(lines), str(d_ans)\n    except Exception:\n        return None\n\n\n# ── Unit conversion ───────────────────────────────────────────────────────────\ndef make_unit_cot(prompt: str):\n    pairs = re.findall(r"([\\d.]+)\\s*(?:[a-zA-Z]+)?\\s+becomes\\s+([\\d.]+)", prompt, re.IGNORECASE)\n    q_match = re.search(r"convert.*?:\\s*([\\d.]+)", prompt, re.IGNORECASE)\n    if not pairs or not q_match:\n        return None\n    try:\n        q_val = float(q_match.group(1))\n        factors = [float(b) / float(a) for a, b in pairs if float(a) != 0]\n        if not factors:\n            return None\n        k = statistics.median(factors)\n        result = round(k * q_val, 2)\n\n        lines = ["Find scale factor k = output/input:"]\n        for a, b in pairs:\n            k_i = round(float(b) / float(a), 4)\n            lines.append(f"  {a} → {b}  k = {b}/{a} = {k_i}")\n        lines.append(f"Median k = {round(k,4)}")\n        lines.append(f"Query: {q_val} × {round(k,4)} = {result}")\n        return "\\n".join(lines), str(result)\n    except Exception:\n        return None\n\n\n# ── Numeral system ────────────────────────────────────────────────────────────\n_ROMAN_VAL = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]\n_ROMAN_SYM = ["M", "CM", "D", "CD", "C", "XC", "L", "XL", "X", "IX", "V", "IV", "I"]\n\n\ndef _to_roman(n: int) -> str:\n    r = ""\n    for v, s in zip(_ROMAN_VAL, _ROMAN_SYM):\n        while n >= v:\n            r += s; n -= v\n    return r\n\n\ndef _parse_numeral_examples(prompt: str):\n    pairs = []\n    for line in prompt.splitlines():\n        m = re.match(r"\\s*(\\S+)\\s*->\\s*(\\S+)", line)\n        if m:\n            pairs.append((m.group(1).strip(), m.group(2).strip()))\n    query = None\n    for line in reversed(prompt.splitlines()):\n        m = re.search(r"(?:number|write)\\s+(\\d+)", line, re.IGNORECASE)\n        if m:\n            query = m.group(1).strip()\n            break\n    if not pairs or not query:\n        return None\n    return pairs, query\n\n\ndef make_numeral_cot(prompt: str):\n    parsed = _parse_numeral_examples(prompt)\n    if not parsed:\n        return None\n    pairs, query = parsed\n\n    is_roman = all(re.match(r"^[IVXLCDMivxlcdm]+$", o) and re.match(r"^\\d+$", i)\n                   for i, o in pairs)\n    if is_roman:\n        try:\n            n = int(query)\n            answer = _to_roman(n)\n            lines = ["Examples confirm standard Roman numerals:"]\n            for i, o in pairs[:3]:\n                lines.append(f"  {i} → {o}  ✓")\n            lines.append(f"Query {n} in Roman = {answer}")\n            return "\\n".join(lines), answer\n        except Exception:\n            return None\n\n    lookup = dict(pairs)\n    if query in lookup:\n        answer = lookup[query]\n        return f"Direct mapping from examples: {query} → {answer}", answer\n\n    return None\n\n\n# ── Cipher ────────────────────────────────────────────────────────────────────\ndef _build_char_map_from_pairs(inputs, outputs):\n    char_map = {}\n    for cipher_sent, plain_sent in zip(inputs, outputs):\n        cwords = cipher_sent.strip().split()\n        pwords = plain_sent.strip().split()\n        if len(cwords) != len(pwords):\n            continue\n        for cw, pw in zip(cwords, pwords):\n            if len(cw) != len(pw):\n                continue\n            for cc, pc in zip(cw, pw):\n                char_map[cc] = pc\n    return char_map\n\n\ndef make_cipher_cot(prompt: str):\n    inputs, outputs, query = _parse_examples(prompt)\n    if not inputs or not query:\n        return None\n\n    query = re.sub(r\'^(?:text|following|for|number)\\s*:?\\s*\', \'\', query, flags=re.IGNORECASE).strip()\n    if not query:\n        return None\n\n    char_map = _build_char_map_from_pairs(inputs, outputs)\n    if not char_map:\n        return None\n\n    result = ""\n    for ch in query:\n        if ch == " ":\n            result += " "\n        elif ch in char_map:\n            result += char_map[ch]\n        else:\n            return None\n    result = result.strip()\n    if not result:\n        return None\n\n    lines = ["Letter substitution cipher — build mapping from examples:"]\n    for k, v in sorted(char_map.items())[:10]:\n        lines.append(f"  {k} → {v}")\n    if len(char_map) > 10:\n        lines.append(f"  ... ({len(char_map)} total mappings)")\n    lines.append(f"Decode \'{query}\':")\n    steps = "  " + " ".join(\n        f"{ch}→{char_map[ch]}" if ch != " " else "[sp]" for ch in query\n    )\n    lines.append(steps)\n    lines.append(f"Result: {result}")\n    return "\\n".join(lines), result\n\n\n# ── Bit manipulation ──────────────────────────────────────────────────────────\ndef make_bit_cot(prompt: str):\n    bit_pairs = re.findall(r"([01]{8})\\s*->\\s*([01]{8})", prompt)\n    query_match = re.search(\n        r"(?:determine|convert|write|now).*?([01]{8})", prompt, re.IGNORECASE | re.DOTALL\n    )\n    if not bit_pairs or not query_match:\n        return None\n    try:\n        ex = [(int(i, 2), int(o, 2)) for i, o in bit_pairs]\n        query_str = query_match.group(1)\n        q = int(query_str, 2)\n    except ValueError:\n        return None\n\n    answer = solve_bit(prompt)\n    if answer is None:\n        return None\n\n    single = _try_single_ops(ex)\n    if single:\n        op_name, param = single\n        label = op_name if not param else f"{op_name} {param}"\n        lines = [f"8-bit operation detected: {label}"]\n        for i_int, o_int in ex[:3]:\n            lines.append(f"  {i_int:08b} → {o_int:08b}  ✓")\n        lines.append(f"Apply to {query_str}: {label}({query_str}) = {answer}")\n        return "\\n".join(lines), answer\n\n    two = _try_two_ops(ex)\n    if two:\n        op1, p1, op2, p2 = two\n        label1 = op1 if not p1 else f"{op1} {p1}"\n        label2 = op2 if not p2 else f"{op2} {p2}"\n        mid_int = _apply_bit_op(q, op1, p1)\n        mid_str = f"{mid_int:08b}"\n        lines = [f"Two-step 8-bit operation: {label1} then {label2}"]\n        for i_int, o_int in ex[:3]:\n            m = _apply_bit_op(i_int, op1, p1)\n            lines.append(f"  {i_int:08b} → {m:08b} → {o_int:08b}  ✓")\n        lines.append(f"Apply to {query_str}:")\n        lines.append(f"  Step 1 {label1}: {query_str} → {mid_str}")\n        lines.append(f"  Step 2 {label2}: {mid_str} → {answer}")\n        return "\\n".join(lines), answer\n\n    lines = ["Per-bit boolean function:"]\n    for i_int, o_int in ex[:4]:\n        lines.append(f"  {i_int:08b} → {o_int:08b}")\n    lines.append(f"Apply pattern to {query_str}: {answer}")\n    return "\\n".join(lines), answer\n\n\n# ── Validation ────────────────────────────────────────────────────────────────\ndef answers_match(computed: str, labeled: str, ptype: str) -> bool:\n    if computed is None:\n        return False\n    if ptype in ("gravity", "unit"):\n        try:\n            return abs(float(computed) - float(labeled)) <= 0.02\n        except Exception:\n            return False\n    elif ptype == "cipher":\n        return str(computed).strip().lower() == str(labeled).strip().lower()\n    else:  # numeral, bit\n        return str(computed).strip().upper() == str(labeled).strip().upper()\n\n\nMAKERS = {\n    "gravity": make_gravity_cot,\n    "unit":    make_unit_cot,\n    "numeral": make_numeral_cot,\n    "cipher":  make_cipher_cot,\n    "bit":     make_bit_cot,\n}\n', encoding='utf-8')
print('wrote makers.py')


In [ ]:
from pathlib import Path
WORK = Path('/kaggle/working/nemotron_challenge')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'build_corpus.py').write_text('"""\nREAL curriculum builder.\nSolves each train row with our verified CoT makers and emits training-ready\nchat records. Every emitted example is VERIFIED (computed answer == gold), so\nthe model only ever sees correct targets.\n\nFor each verified row we emit two reasoning records (never answer-only, so the\nmodel always learns to think before boxing):\n  - real_trace   : the full verified CoT + box    (the complete reasoning path)\n  - real_concise : rule line + result line + box  (the same reasoning, short form)\n\nOutput: real_curriculum.jsonl  (messages format, ready for SFT)\n"""\nimport csv, json\nfrom pathlib import Path\nfrom collections import defaultdict\n\nimport makers\n\nDATA = next(p for p in [Path("data/train.csv"),\n            Path("data/nvidia-nemotron-model-reasoning-challenge/train.csv")] if p.exists())\nOUT = Path("real_curriculum.jsonl")\nSUFFIX = "Please put your final answer inside \\\\boxed{}."\n\n\ndef make_record(rid, bucket, source, prompt, response):\n    return {\n        "id": rid,\n        "bucket": bucket,\n        "source": source,\n        "messages": [\n            {"role": "user", "content": prompt.strip() + "\\n" + SUFFIX},\n            {"role": "assistant", "content": response},\n        ],\n    }\n\n\ndef concise_trace(trace):\n    """Reduce a full CoT to rule + result: the first (rule-identifying) line and\n    the last (application) line. Keeps real reasoning in short form so the\n    record is never answer-only."""\n    lines = [ln.strip() for ln in trace.splitlines() if ln.strip()]\n    if len(lines) <= 2:\n        return trace.strip()\n    return lines[0] + "\\n" + lines[-1]\n\n\ndef main():\n    rows = list(csv.DictReader(open(DATA, encoding="utf-8")))\n    records = []\n    verified = 0\n    weak_bit = 0\n    by_bucket = defaultdict(int)\n\n    for row in rows:\n        t = makers.detect_type(row["prompt"])\n        if t not in makers.MAKERS:\n            continue\n        res = makers.MAKERS[t](row["prompt"])\n        if res is None:\n            continue\n        trace, ans = res\n        if not makers.answers_match(ans, row["answer"], t):\n            continue\n        # Drop bit rows whose only trace is the weak per-bit-boolean table-dump\n        # (no clean op recovered): it teaches no transferable method. Better to\n        # learn bit from the strong single/two-op reals + synthetic coverage.\n        if t == "bit" and "Per-bit boolean function:" in trace:\n            weak_bit += 1\n            continue\n\n        verified += 1\n        by_bucket[t] += 1\n        gold = row["answer"].strip()\n        box = f"\\\\boxed{{{gold}}}"\n        full = trace.strip() + f"\\n\\nFinal answer: {box}"\n        records.append(make_record(f"{row[\'id\']}:trace", t, "real_trace",\n                                    row["prompt"], full))\n        concise = concise_trace(trace).strip() + f"\\n\\nFinal answer: {box}"\n        records.append(make_record(f"{row[\'id\']}:concise", t, "real_concise",\n                                    row["prompt"], concise))\n\n    with open(OUT, "w", encoding="utf-8") as f:\n        for r in records:\n            f.write(json.dumps(r, ensure_ascii=False) + "\\n")\n\n    print(f"verified rows: {verified}  ->  {len(records)} records  -> {OUT}")\n    print(f"dropped weak-bit (per-bit boolean only): {weak_bit}")\n    for t in ["bit", "gravity", "unit", "numeral", "cipher", "symbol"]:\n        print(f"  {t:<9} {by_bucket[t]}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('wrote build_corpus.py')


In [ ]:
from pathlib import Path
WORK = Path('/kaggle/working/nemotron_challenge')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'synth.py').write_text('"""\nSYNTHETIC generator.\nWe control the rule, so every generated example is verifiable and ships with a\ncorrect reasoning trace. Prompts mirror the real train templates exactly so the\nmodel sees consistent formatting at train/test time.\n\nBuckets generated: bit, gravity, unit, numeral, cipher.\n(symbol is intentionally omitted: real symbol_transform is not a clean per-symbol\nsubstitution, so synthetic symbol would not match the hidden rule family.)\n\nWeighted toward the failure-prone buckets (cipher especially) by default.\nOutput: synth_curriculum.jsonl\n"""\nimport json, random, string\nfrom pathlib import Path\n\nfrom solvers import _apply_bit_op  # reuse our verified bit primitive\n\nSUFFIX = "Please put your final answer inside \\\\boxed{}."\nALPHABET = string.ascii_lowercase\nWORD_POOL = [\n    "queen", "dragon", "castle", "wizard", "forest", "silver", "mirror", "palace",\n    "secret", "garden", "bridge", "market", "planet", "rocket", "harbor", "magnet",\n    "window", "stone", "cable", "river", "knight", "shadow", "flame", "jewel",\n    "puzzle", "voyage", "quartz", "frozen", "bishop", "candle", "dwarf", "lucky",\n    "myth", "vex", "jazz", "box", "fjord", "glyph",\n]\n\n\ndef _box(ans):\n    return f"\\\\boxed{{{ans}}}"\n\n\ndef make_record(bucket, prompt, trace, answer, idx):\n    full = trace.strip() + f"\\n\\nFinal answer: {_box(answer)}"\n    return {\n        "id": f"synth:{bucket}:{idx}",\n        "bucket": bucket,\n        "source": "synthetic_trace",\n        "messages": [\n            {"role": "user", "content": prompt.strip() + "\\n" + SUFFIX},\n            {"role": "assistant", "content": full},\n        ],\n    }\n\n\n# ── bit ─────────────────────────────────────────────────────────────────────\nBIT_HEADER = ("In Alice\'s Wonderland, a secret bit manipulation rule transforms 8-bit "\n              "binary numbers. The transformation involves operations like bit shifts, "\n              "rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.")\n\ndef _bit_prompt(ex, q):\n    lines = [BIT_HEADER, "", "Here are some examples of input -> output:"]\n    lines += [f"{i:08b} -> {o:08b}" for i, o in ex]\n    lines += ["", f"Now, determine the output for: {q:08b}"]\n    return "\\n".join(lines)\n\n\ndef gen_bit(rng):\n    # ~30% two-step rules so the model also learns composed transforms (these\n    # are recoverable by solve_bit\'s _try_two_ops, so curate keeps them).\n    if rng.random() < 0.3:\n        return _gen_bit_two_step(rng)\n    rules = (\n        [("NOT", 0, "invert every bit (NOT)")]\n        + [("REV", 0, "reverse the bit order")]\n        + [("ROL", k, f"rotate left by {k}") for k in (1, 2, 3)]\n        + [("ROR", k, f"rotate right by {k}") for k in (1, 2, 3)]\n        + [("XOR", m, f"XOR with mask {m:08b}") for m in (rng.randint(1, 255),)]\n        + [("AND", m, f"AND with mask {m:08b}") for m in (rng.randint(1, 255),)]\n        + [("OR", m, f"OR with mask {m:08b}") for m in (rng.randint(1, 255),)]\n        + [("SHL", k, f"shift left by {k}") for k in (1, 2)]\n        + [("SHR", k, f"shift right by {k}") for k in (1, 2)]\n    )\n    op, param, desc = rng.choice(rules)\n    vals = rng.sample(range(256), 9)\n    ex = [(v, _apply_bit_op(v, op, param)) for v in vals[:8]]\n    q = vals[8]\n    ans = f"{_apply_bit_op(q, op, param):08b}"\n    trace = (f"Comparing inputs to outputs, the rule is to {desc}. "\n             f"Applying it to {q:08b} gives {ans}.")\n    return _bit_prompt(ex, q), trace, ans\n\n\n# op1 must lie in solve_bit\'s _try_two_ops candidate set (NOT/REV/ROL/ROR) so\n# the composed rule stays recoverable on re-verify.\n_TWO_OP1 = (\n    [("NOT", 0, "invert every bit (NOT)"), ("REV", 0, "reverse the bit order")]\n    + [("ROL", k, f"rotate left by {k}") for k in (1, 2, 3)]\n    + [("ROR", k, f"rotate right by {k}") for k in (1, 2, 3)]\n)\n\n\ndef _gen_bit_two_step(rng):\n    op1, p1, desc1 = rng.choice(_TWO_OP1)\n    op2, p2, desc2 = rng.choice(\n        [("NOT", 0, "invert every bit (NOT)"), ("REV", 0, "reverse the bit order")]\n        + [("ROL", k, f"rotate left by {k}") for k in (1, 2)]\n        + [("ROR", k, f"rotate right by {k}") for k in (1, 2)]\n        + [("XOR", m, f"XOR with mask {m:08b}") for m in (rng.randint(1, 255),)]\n    )\n    apply2 = lambda v: _apply_bit_op(_apply_bit_op(v, op1, p1), op2, p2)\n    vals = rng.sample(range(256), 9)\n    ex = [(v, apply2(v)) for v in vals[:8]]\n    if all(i == o for i, o in ex):       # composed to identity -> retry\n        return _gen_bit_two_step(rng)\n    q = vals[8]\n    mid = _apply_bit_op(q, op1, p1)\n    ans = f"{apply2(q):08b}"\n    trace = (f"Comparing inputs to outputs, the rule is two steps: first {desc1}, "\n             f"then {desc2}. Applying to {q:08b}: step 1 gives {mid:08b}, "\n             f"step 2 gives {ans}.")\n    return _bit_prompt(ex, q), trace, ans\n\n\n# ── gravity ───────────────────────────────────────────────────────────────────\ndef gen_gravity(rng):\n    g = round(rng.uniform(4.0, 25.0), 2)\n    ts = [round(rng.uniform(1.0, 5.0), 2) for _ in range(6)]\n    obs = [(t, round(0.5 * g * t * t, 2)) for t in ts[:5]]\n    qt = ts[5]\n    ans = f"{0.5 * g * qt * qt:.2f}"\n    lines = ["In Alice\'s Wonderland, the gravitational constant has been secretly changed. "\n             "Here are some example observations:"]\n    lines += [f"For t = {t}s, distance = {d} m" for t, d in obs]\n    lines += [f"Now, determine the falling distance for t = {qt}s given d = 0.5*g*t^2."]\n    trace = (f"From d = 0.5*g*t^2 we get g = 2d/t^2. Each observation gives g ≈ {g}. "\n             f"So for t = {qt}s, d = 0.5*{g}*{qt}^2 = {ans}.")\n    return "\\n".join(lines), trace, ans\n\n\n# ── unit ──────────────────────────────────────────────────────────────────────\ndef gen_unit(rng):\n    k = round(rng.uniform(0.3, 3.0), 4)\n    xs = [round(rng.uniform(5.0, 40.0), 2) for _ in range(rng.choice([4, 5, 6]))]\n    obs = [(x, round(k * x, 2)) for x in xs[:-1]]\n    qx = xs[-1]\n    ans = f"{k * qx:.2f}"\n    lines = ["In Alice\'s Wonderland, a secret unit conversion is applied to measurements. For example:"]\n    lines += [f"{x} m becomes {y}" for x, y in obs]\n    lines += [f"Now, convert the following measurement: {qx} m"]\n    trace = (f"The scale factor is k = output/input ≈ {k}. "\n             f"So {qx} × {k} = {ans}.")\n    return "\\n".join(lines), trace, ans\n\n\n# ── numeral (decimal -> roman) ─────────────────────────────────────────────────\n_RV = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]\n_RS = ["M", "CM", "D", "CD", "C", "XC", "L", "XL", "X", "IX", "V", "IV", "I"]\n\ndef _to_roman(n):\n    r = ""\n    for v, s in zip(_RV, _RS):\n        while n >= v:\n            r += s; n -= v\n    return r\n\ndef gen_numeral(rng):\n    nums = rng.sample(range(1, 200), 5)\n    q = rng.randint(1, 200)\n    ans = _to_roman(q)\n    lines = ["In Alice\'s Wonderland, numbers are secretly converted into a different numeral system. "\n             "Some examples are given below:"]\n    lines += [f"{n} -> {_to_roman(n)}" for n in nums]\n    lines += [f"Now, write the number {q} in the Wonderland numeral system."]\n    trace = (f"The examples are standard Roman numerals. Converting {q}: {ans}.")\n    return "\\n".join(lines), trace, ans\n\n\n# ── cipher (random monoalphabetic, full query coverage) ────────────────────────\ndef gen_cipher(rng):\n    perm = list(ALPHABET)\n    rng.shuffle(perm)\n    enc = {p: c for p, c in zip(ALPHABET, perm)}     # plain -> cipher\n    encode = lambda s: "".join(enc.get(ch, ch) for ch in s)\n\n    ex_plain = [" ".join(rng.sample(WORD_POOL, rng.choice([3, 4]))) for _ in range(5)]\n    seen = set("".join(ex_plain).replace(" ", ""))\n    candidates = [w for w in WORD_POOL if set(w) <= seen]\n    if len(candidates) < 2:\n        return gen_cipher(rng)  # retry; rare\n    q_plain = " ".join(rng.sample(candidates, rng.choice([2, 3])))\n\n    lines = ["In Alice\'s Wonderland, secret encryption rules are used on text. Here are some examples:"]\n    lines += [f"{encode(p)} -> {p}" for p in ex_plain]\n    lines += [f"Now, decrypt the following text: {encode(q_plain)}"]\n    # trace: reconstruct cipher->plain map from a few pairs\n    dec_pairs = sorted({(enc[p], p) for p in seen})[:8]\n    mapping = ", ".join(f"{c}→{p}" for c, p in dec_pairs)\n    trace = (f"Each cipher letter maps to a fixed plaintext letter. "\n             f"From the examples: {mapping}, .... "\n             f"Decoding \'{encode(q_plain)}\' gives \'{q_plain}\'.")\n    return "\\n".join(lines), trace, q_plain\n\n\nGENERATORS = {\n    "bit": gen_bit,\n    "gravity": gen_gravity,\n    "unit": gen_unit,\n    "numeral": gen_numeral,\n    "cipher": gen_cipher,\n}\n\n# default allocation — weighted toward failure-prone buckets. bit is raised to\n# backfill the real bit rows now dropped for weak (per-bit-boolean) traces.\nDEFAULT_ALLOC = {"cipher": 3000, "bit": 3000, "numeral": 800, "gravity": 800, "unit": 800}\nOUT = Path("synth_curriculum.jsonl")\n\n\ndef main(alloc=None, seed=3407):\n    alloc = alloc or DEFAULT_ALLOC\n    rng = random.Random(seed)\n    records = []\n    for bucket, n in alloc.items():\n        for i in range(n):\n            prompt, trace, ans = GENERATORS[bucket](rng)\n            records.append(make_record(bucket, prompt, trace, ans, i))\n    rng.shuffle(records)\n    with open(OUT, "w", encoding="utf-8") as f:\n        for r in records:\n            f.write(json.dumps(r, ensure_ascii=False) + "\\n")\n    print(f"synthetic records: {len(records)} -> {OUT}")\n    for b, n in alloc.items():\n        print(f"  {b:<9} {n}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('wrote synth.py')


In [ ]:
from pathlib import Path
WORK = Path('/kaggle/working/nemotron_challenge')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'curate.py').write_text('"""\nDATA CURATION + assembly.\nTakes the raw real + synthetic sets and produces a clean, accurate, training-ready\ncurriculum. Every surviving record passes the SAME accuracy bar.\n\nStages (each reports how many it drops):\n1. format validation     - non-empty turns, parseable \\boxed{}\n2. trace-quality filter   - drop bit "per-bit boolean" table-dump traces\n3. round-trip re-verify   - re-solve the prompt independently; boxed answer must match\n4. dedupe                 - exact (prompt,response); cap identical synthetic prompts\n5. length cap             - drop records longer than the training context allows\n6. assemble               - oversample hard buckets (bit, cipher), shuffle\n\nInput : real_curriculum.jsonl, synth_curriculum.jsonl\nOutput: curated_curriculum.jsonl\n"""\nimport json, re, random\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\n\nimport makers\nfrom solvers import solve_bit  # stronger bit coverage than the maker\n\nREAL = Path("real_curriculum.jsonl")\nSYNTH = Path("synth_curriculum.jsonl")\nOUT = Path("curated_curriculum.jsonl")\n\nCHAR_CAP = 20000             # ~5k tokens, safely under the 6144 train context\nMAX_SYNTH_PROMPT_DUPES = 1   # keep at most N synthetic records per identical prompt\nHARD_OVERSAMPLE = {"bit": 2, "cipher": 2}\nSEED = 3407\n\nBOX = re.compile(r"\\\\boxed\\{([^}]*)\\}")\n\n\ndef boxed(text):\n    m = BOX.findall(text or "")\n    return m[-1].strip() if m else None\n\n\ndef user_prompt(rec):\n    return rec["messages"][0]["content"]\n\n\ndef base_prompt(rec):\n    # strip the trailing boxed-answer suffix to recover the raw problem text\n    return user_prompt(rec).rsplit("\\nPlease put your final answer", 1)[0]\n\n\n# ---- stage 3: independent re-solver (cached per raw prompt) ----\n_cache = {}\n\ndef resolve_answer(rec):\n    bp = base_prompt(rec)\n    if bp in _cache:\n        return _cache[bp]\n    t = makers.detect_type(bp)\n    ans = None\n    try:\n        if t == "bit":\n            ans = solve_bit(bp)\n        elif t in makers.MAKERS:\n            res = makers.MAKERS[t](bp)\n            ans = res[1] if res else None\n    except Exception:\n        ans = None\n    _cache[bp] = (t, ans)\n    return t, ans\n\n\ndef main():\n    real = [json.loads(l) for l in REAL.read_text(encoding="utf-8").splitlines() if l.strip()]\n    synth = [json.loads(l) for l in SYNTH.read_text(encoding="utf-8").splitlines() if l.strip()]\n    drops = Counter()\n    kept = []\n\n    for rec, is_synth in [(r, False) for r in real] + [(r, True) for r in synth]:\n        u = user_prompt(rec); a = rec["messages"][1]["content"]\n        # 1. format\n        if not u.strip() or not a.strip():\n            drops["empty"] += 1; continue\n        ans = boxed(a)\n        if ans is None:\n            drops["no_box"] += 1; continue\n        # 2. trace quality\n        if rec["bucket"] == "bit" and "Per-bit boolean function:" in a:\n            drops["weak_bit_trace"] += 1; continue\n        # 3. round-trip verify\n        t, recovered = resolve_answer(rec)\n        if recovered is None or not makers.answers_match(recovered, ans, t):\n            drops["reverify_fail"] += 1; continue\n        # 5. length (do cheap char check here)\n        if len(u) + len(a) > CHAR_CAP:\n            drops["too_long"] += 1; continue\n        kept.append(rec)\n\n    # 4. dedupe exact (prompt,response) + cap synthetic prompt repeats\n    seen_pair, synth_prompt_count, deduped = set(), defaultdict(int), []\n    for rec in kept:\n        key = (user_prompt(rec), rec["messages"][1]["content"])\n        if key in seen_pair:\n            drops["dup_exact"] += 1; continue\n        seen_pair.add(key)\n        if rec["source"] == "synthetic_trace":\n            p = user_prompt(rec)\n            if synth_prompt_count[p] >= MAX_SYNTH_PROMPT_DUPES:\n                drops["dup_synth_prompt"] += 1; continue\n            synth_prompt_count[p] += 1\n        deduped.append(rec)\n\n    # 6. assemble: oversample hard real buckets, then shuffle\n    rng = random.Random(SEED)\n    final = []\n    for rec in deduped:\n        reps = HARD_OVERSAMPLE.get(rec["bucket"], 1) if rec["source"].startswith("real") else 1\n        final.extend([rec] * reps)\n    rng.shuffle(final)\n\n    with open(OUT, "w", encoding="utf-8") as f:\n        for rec in final:\n            f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n\n    # ---- report ----\n    print(f"loaded:   real={len(real)}  synth={len(synth)}  total={len(real)+len(synth)}")\n    print(f"dropped:  {dict(drops)}")\n    print(f"after curation+dedupe: {len(deduped)}")\n    print(f"after oversample (final): {len(final)}  -> {OUT}\\n")\n    print("final by source:", dict(Counter(r["source"] for r in final)))\n    print("final by bucket:", dict(Counter(r["bucket"] for r in final)))\n    chars = sum(len(user_prompt(r)) + len(r["messages"][1]["content"]) for r in final)\n    print(f"~chars: {chars:,}  (~{chars//4:,} tokens)")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('wrote curate.py')


In [ ]:
from pathlib import Path
WORK = Path('/kaggle/working/nemotron_challenge')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'build_thinking_curriculum.py').write_text('"""\nRewrite the curated curriculum into the host\'s reasoning ("thinking") format.\n\nThe scoring backend renders test prompts with the Nemotron chat template in its\ndefault reasoning mode, so the generation prompt ends with:\n\n    <SPECIAL_14>Assistant\\n<think>\\n\n\ni.e. the model is primed *inside* an open <think> block. Our original targets put\nthe reasoning as plain text and the answer inline, which would not line up with\nthat primed prefix (the prompt render would not be a token prefix of the full\nrender, breaking completion masking).\n\nThis script moves each record\'s reasoning inside an explicit <think>...</think>\nblock and places the final \\boxed{} answer *after* the block:\n\n    <think>\n    {reasoning}\n    </think>\n\n    \\boxed{X}\n\nso the assistant turn begins with "<think>\\n", matching the primed prefix, and\nthe answer is unambiguous for \\boxed{} extraction. Every upstream record now\ncarries a real reasoning trace (build_corpus emits real_trace + real_concise,\nsynth emits a trace), so an empty think block is unexpected and flagged below.\n\nInput : curated_curriculum.jsonl\nOutput: curriculum_thinking.jsonl\n"""\nimport json\nimport re\n\nSRC = "curated_curriculum.jsonl"\nDST = "curriculum_thinking.jsonl"\n\n# trailing lead-in phrases to drop from the reasoning once the boxed answer is\n# pulled out (avoids a dangling "The answer is" before the closing </think>)\nTRAILING = re.compile(\n    r"\\s*(the\\s+answer\\s+is|final\\s+answer|therefore|thus|so|hence|answer)\\s*[:=]?\\s*$",\n    re.IGNORECASE,\n)\n\n\ndef last_boxed_span(s):\n    """Span (start, end) of the last balanced \\\\boxed{...}; end is past the brace."""\n    idx = s.rfind(r"\\boxed{")\n    if idx == -1:\n        return None\n    i = idx + len(r"\\boxed{")\n    depth = 1\n    while i < len(s) and depth:\n        depth += {"{": 1, "}": -1}.get(s[i], 0)\n        i += 1\n    return (idx, i) if depth == 0 else None\n\n\ndef to_thinking(content):\n    sp = last_boxed_span(content)\n    if sp is None:\n        raise ValueError("no balanced \\\\boxed{} in: " + content[:80])\n    answer = content[sp[0]:sp[1]]\n    reasoning = TRAILING.sub("", content[:sp[0]].rstrip()).rstrip()\n    inner = f"\\n{reasoning}\\n" if reasoning else "\\n"\n    return f"<think>{inner}</think>\\n\\n{answer}"\n\n\ndef main():\n    n = empty = 0\n    with open(SRC, encoding="utf-8") as fin, open(DST, "w", encoding="utf-8") as fout:\n        for line in fin:\n            if not line.strip():\n                continue\n            rec = json.loads(line)\n            msgs = rec["messages"]\n            assert msgs[-1]["role"] == "assistant", "last turn must be assistant"\n            new = to_thinking(msgs[-1]["content"])\n            if new.startswith("<think>\\n</think>"):\n                empty += 1\n            msgs[-1]["content"] = new\n            fout.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n            n += 1\n    print(f"wrote {n} records to {DST} ({empty} empty-think, {n - empty} with reasoning)")\n    if empty:\n        print(f"WARNING: {empty} empty-think records remain — every record should "\n              f"carry reasoning now; check build_corpus/synth output.")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('wrote build_thinking_curriculum.py')


## Build the curriculum

Build and validate the training curriculum directly from the attached
competition data.


In [ ]:
import shutil
from pathlib import Path

WORK = Path("/kaggle/working/nemotron_challenge")
candidates = sorted(Path("/kaggle/input").glob("**/train.csv"))
assert candidates, "No train.csv found. Attach the competition data input."

preferred = [
    p for p in candidates
    if "nvidia-nemotron-model-reasoning-challenge" in str(p).lower()
]
train_csv = preferred[0] if preferred else candidates[0]

data_dir = WORK / "data"
data_dir.mkdir(exist_ok=True)
shutil.copyfile(train_csv, data_dir / "train.csv")
print("using train.csv:", train_csv)
print("copied to:", data_dir / "train.csv")


In [ ]:
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

WORK = Path("/kaggle/working/nemotron_challenge")
for script in [
    "build_corpus.py",
    "synth.py",
    "curate.py",
    "build_thinking_curriculum.py",
]:
    print(f"\n=== {script} ===", flush=True)
    subprocess.run([sys.executable, str(WORK / script)], cwd=WORK, check=True)

curriculum = WORK / "curriculum_thinking.jsonl"
assert curriculum.exists(), "final curriculum was not written"

records = []
with curriculum.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

bad = [
    r["id"] for r in records
    if not (
        r["messages"][-1]["content"].startswith("<think>")
        and "</think>" in r["messages"][-1]["content"]
        and "\\boxed{" in r["messages"][-1]["content"]
    )
]
assert not bad, f"bad thinking-format records: {bad[:5]}"

print("\nfinal records:", len(records))
print("by bucket:", dict(Counter(r["bucket"] for r in records)))
print("by source:", dict(Counter(r["source"] for r in records)))
print("curriculum path:", curriculum)


## Fine-tune and package

Read the curriculum built above, fine-tune the rank-32 LoRA adapter with
prompt-masked loss, and package `submission.zip` with the adapter files the
competition scorer expects.


In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
)

# Make stdout/stderr line-buffered so Kaggle streams logs live.
import sys as _sys
try:
    _sys.stdout.reconfigure(line_buffering=True)
    _sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
]


def find_base_model():
    for raw in BASE_MODEL_CANDIDATES:
        path = Path(raw)
        if (path / "config.json").exists():
            return str(path)

    # Fallback: search attached Kaggle inputs for a Nemotron model directory.
    root = Path("/kaggle/input")
    if root.exists():
        for cfg in sorted(root.glob("**/config.json")):
            parent = cfg.parent
            name = str(parent).lower()
            if "nemotron" in name and ("30b" in name or "nano" in name):
                return str(parent)

    raise FileNotFoundError(
        "Could not find the Nemotron base model under /kaggle/input. "
        "Attach the Kaggle model input metric/nemotron-3-nano-30b-a3b-bf16 "
        "and run a cell with `!find /kaggle/input -name config.json | head -20` "
        "to inspect the mounted path."
    )


BASE_MODEL = find_base_model()
print("base model:", BASE_MODEL)
# Canonical HF id, written into the packaged adapter_config (the kaggle input
# path above is local to this kernel and won't exist on the scoring backend).
BASE_MODEL_HF = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

# The Nemotron chat template defaults to reasoning ("thinking") mode ON and
# toggles it only via "/think" | "/no_think" in the message content -- it does
# NOT read an enable_thinking kwarg.
# match (assistant turn starts with "<think>\n"; see build_thinking_curriculum.py),
# so we render with the template default and pass no kwarg.
ENABLE_THINKING = None

WARMSTART_ADAPTER = None
# WARMSTART_ADAPTER = "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20"

# Seed adapter_config.json from a known-good reference so Kaggle's vLLM backend
# loads the adapter cleanly. Set to "" to skip if not attached.
REFERENCE_SUBMISSION_ZIP = "/kaggle/input/nvidia-nemotron-all-linear/reference/submission.zip"

OUTPUT_DIR = Path("/kaggle/working/adapter")
SUBMISSION_ZIP = Path("/kaggle/working/submission.zip")

LORA_R = 32            # competition cap is rank 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = "all-linear"
MAX_LENGTH = 4096
MAX_STEPS = 450        # tune to the budget; raise if eval-loss is still falling
LEARNING_RATE = 2e-4
BATCH = 1
GRAD_ACCUM = 16
SEED = 3407

# Held-out eval-loss + task-accuracy probe (keep probe config small).
VAL_PER_BUCKET = 40          # held-out rows per bucket for eval-loss
EVAL_STEPS = 50              # eval-loss cadence
PROBE_PER_BUCKET = 3         # generated examples per bucket for task accuracy
PROBE_AT_STEPS = [0, MAX_STEPS]        # baseline + final only (no mid probe)
PROBE_MAX_NEW_TOKENS = 256   # generation runs WITHOUT a KV cache on this model,
                             # so keep this small -- it dominates probe cost.
METRICS_JSON = Path("/kaggle/working/training_metrics.json")

# ---------------------------------------------------------------------------
# Load curriculum built earlier in this same notebook
# ---------------------------------------------------------------------------
CURRICULUM = Path("/kaggle/working/nemotron_challenge/curriculum_thinking.jsonl")
assert CURRICULUM.exists(), (
    "curriculum_thinking.jsonl is missing. Run the data-build cells above before training."
)
rows = [json.loads(line) for line in CURRICULUM.open(encoding="utf-8") if line.strip()]
print(f"curriculum: {CURRICULUM}  ({len(rows)} records)")

# ---------------------------------------------------------------------------
# Tokenize with prompt masking (only assistant tokens contribute to the loss)
# ---------------------------------------------------------------------------
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=False)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token


def render(messages, add_generation_prompt):
    kwargs = {}
    if ENABLE_THINKING is not None:
        kwargs["enable_thinking"] = ENABLE_THINKING
    return tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt, **kwargs
    )


def encode(row):
    # Prompt = everything up to (not incl.) the assistant turn, WITH the
    # generation prompt; full = the whole conversation. The prompt render must
    # be an exact token prefix of the full render or completion masking is wrong.
    prompt_ids = tok(
        render(row["messages"][:-1], True),
        add_special_tokens=False, truncation=True, max_length=MAX_LENGTH,
    )["input_ids"]
    full_ids = tok(
        render(row["messages"], False),
        add_special_tokens=False, truncation=True, max_length=MAX_LENGTH,
    )["input_ids"]
    assert full_ids[: len(prompt_ids)] == prompt_ids, (
        "chat-template prefix mismatch: the prompt render is not a token prefix "
        "of the full render, so masking would be misaligned. Check ENABLE_THINKING "
        "and add_generation_prompt handling for this tokenizer."
    )
    labels = ([-100] * len(prompt_ids) + full_ids[len(prompt_ids):])[: len(full_ids)]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


# Smoke test: render the first record and show where the prompt ends / the
# completion begins. This runs on CPU before the model loads, so a template
# mismatch (e.g. the primed "<think>" prefix) fails here, not 30 min into GPU.
_p = render(rows[0]["messages"][:-1], True)
_f = render(rows[0]["messages"], False)
print("=== render smoke test ===")
print("PROMPT render (last 80 chars):", repr(_p[-80:]))
print("FULL is prefixed by PROMPT:", _f.startswith(_p))
print("COMPLETION (first 80 chars):", repr(_f[len(_p):len(_p) + 80]))

# Held-out split (stratified by bucket), seeded, removed from train to avoid leak.
import random as _random
from collections import defaultdict as _dd

_by_bucket = _dd(list)
for _r in rows:
    _by_bucket[_r.get("bucket", "?")].append(_r)

_rng = _random.Random(SEED)
val_rows, probe_rows, train_rows = [], [], []
for _b, _items in _by_bucket.items():
    _shuf = _items[:]
    _rng.shuffle(_shuf)
    _val = _shuf[:VAL_PER_BUCKET]
    val_rows.extend(_val)
    probe_rows.extend(_val[:PROBE_PER_BUCKET])
    train_rows.extend(_shuf[VAL_PER_BUCKET:])
_rng.shuffle(train_rows)
print(f"split: train={len(train_rows)}  val(eval-loss)={len(val_rows)}  "
      f"probe(task-acc)={len(probe_rows)}")

data = [encode(r) for r in train_rows]
val_data = [encode(r) for r in val_rows]


class ListDataset(torch.utils.data.Dataset):
    def __init__(self, d):
        self.d = d

    def __len__(self):
        return len(self.d)

    def __getitem__(self, i):
        return self.d[i]


class PadCollator:
    def __call__(self, feats):
        m = max(len(x["input_ids"]) for x in feats)
        input_ids, attn, labels = [], [], []
        for x in feats:
            pad = m - len(x["input_ids"])
            input_ids.append(x["input_ids"] + [tok.pad_token_id] * pad)
            attn.append(x["attention_mask"] + [0] * pad)
            labels.append(x["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attn),
            "labels": torch.tensor(labels),
        }


# ---------------------------------------------------------------------------
# Load 30B in BF16 + attach LoRA
# ---------------------------------------------------------------------------
print("BF16 LoRA")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False

if WARMSTART_ADAPTER:
    print("warmstart from", WARMSTART_ADAPTER)
    model = PeftModel.from_pretrained(model, WARMSTART_ADAPTER, is_trainable=True)
else:
    print("training LoRA from scratch")
    model = get_peft_model(
        model,
        LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=TARGET_MODULES,
            task_type="CAUSAL_LM",
        ),
    )
model.print_trainable_parameters()

# ---------------------------------------------------------------------------
# Observability: task-accuracy probe + structured metrics logger
# ---------------------------------------------------------------------------
import re as _re
import time as _time
from transformers import TrainerCallback

_BOX_RE = _re.compile(r"\\boxed\{([^{}]*)\}")


def _gold(rec):
    m = _BOX_RE.findall(rec["messages"][-1]["content"])
    return m[-1].strip() if m else None


def _answers_match(pred, gold, bucket):
    if pred is None or gold is None:
        return False
    if bucket in ("gravity", "unit"):
        try:
            return abs(float(pred) - float(gold)) <= 0.02
        except ValueError:
            return False
    if bucket == "cipher":
        return pred.strip().lower() == gold.strip().lower()
    return pred.strip().upper() == gold.strip().upper()


@torch.no_grad()
def task_accuracy(tag, step):
    """Greedy-generate on the held-out probe; report per-bucket answer accuracy."""
    was_training = model.training
    prev_cache = model.config.use_cache
    # Generation needs the KV cache and is incompatible with gradient
    # checkpointing; disable it defensively and restore afterwards. (HF would
    # otherwise silently force use_cache=False and make the probe very slow.)
    gc_on = bool(getattr(model, "is_gradient_checkpointing", False))
    model.eval()
    model.config.use_cache = True
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass
    dev = next(model.parameters()).device
    per, correct, total = {}, 0, 0
    t0 = _time.time()
    for rec in probe_rows:
        b = rec.get("bucket", "?")
        try:
            ids = tok(render(rec["messages"][:-1], True), return_tensors="pt",
                      add_special_tokens=False).to(dev)
            out = model.generate(**ids, max_new_tokens=PROBE_MAX_NEW_TOKENS,
                                  do_sample=False, pad_token_id=tok.pad_token_id)
            gen = tok.decode(out[0][ids["input_ids"].shape[1]:],
                             skip_special_tokens=True)
            found = _BOX_RE.findall(gen)
            pred = found[-1].strip() if found else None
            ok = _answers_match(pred, _gold(rec), b)
        except Exception as e:
            print(f"[probe {tag}] generate failed on {b}: {e}", flush=True)
            ok = False
        c, t = per.get(b, (0, 0))
        per[b] = (c + int(ok), t + 1)
        correct += int(ok); total += 1
    acc = correct / max(total, 1)
    detail = " ".join(f"{b}={c}/{t}" for b, (c, t) in sorted(per.items()))
    print(f"[probe {tag} step={step}] acc={acc:.3f} ({correct}/{total})  "
          f"{detail}  ({_time.time()-t0:.0f}s)", flush=True)
    # restore training state
    model.config.use_cache = prev_cache
    if gc_on:
        try:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
    if was_training:
        model.train()
    return {"tag": tag, "step": step, "acc": acc,
            "per_bucket": {b: list(v) for b, v in per.items()}}


class MetricsLogger(TrainerCallback):
    def __init__(self):
        self.t0 = None
        self.history = []
        self.probes = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if self.t0 is None:
            self.t0 = _time.time()
        elapsed = _time.time() - self.t0
        step = state.global_step
        rec = {"step": step, "elapsed_s": round(elapsed, 1)}
        for k in ("loss", "eval_loss", "grad_norm", "learning_rate"):
            if k in logs:
                rec[k] = logs[k]
        self.history.append(rec)
        sps = step / elapsed if elapsed > 0 else 0.0
        pct = 100 * step / max(args.max_steps, 1)
        eta_m = (args.max_steps - step) / sps / 60 if sps > 0 else 0.0
        parts = [f"step {step}/{args.max_steps} ({pct:.0f}%)"]
        if "loss" in logs: parts.append(f"loss={logs['loss']:.4f}")
        if "eval_loss" in logs: parts.append(f"eval_loss={logs['eval_loss']:.4f}")
        if "grad_norm" in logs: parts.append(f"gnorm={logs['grad_norm']:.2f}")
        if "learning_rate" in logs: parts.append(f"lr={logs['learning_rate']:.2e}")
        parts.append(f"{sps:.2f} step/s  {elapsed:.0f}s  ETA {eta_m:.0f}m")
        print("[train] " + "  ".join(parts), flush=True)

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        if step in PROBE_AT_STEPS and step not in (0, args.max_steps):
            self.probes.append(task_accuracy("mid", step))


# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
metrics = MetricsLogger()

# Baseline task accuracy BEFORE any training (step 0): the reference to beat.
if 0 in PROBE_AT_STEPS:
    metrics.probes.append(task_accuracy("baseline", 0))

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        max_steps=MAX_STEPS,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        per_device_eval_batch_size=1,
        save_strategy="no",      # don't write optimizer/checkpoint files; the
                                 # adapter is saved via model.save_pretrained below.
        bf16=True,
        optim="adamw_torch",
        gradient_checkpointing=True,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        report_to=[],
        disable_tqdm=True,
        remove_unused_columns=False,
        seed=SEED,
    ),
    train_dataset=ListDataset(data),
    eval_dataset=ListDataset(val_data),
    data_collator=PadCollator(),
    callbacks=[metrics],
)
trainer.train()
model.save_pretrained(OUTPUT_DIR)

# ---------------------------------------------------------------------------
# Package submission.zip FIRST (adapter_config.json + adapter_model.safetensors)
# Done right after saving the adapter so the submission is secured before the
# slower final probe runs -- a probe failure or session cut can't lose it.
# ---------------------------------------------------------------------------
pkg = Path("/kaggle/working/package")
shutil.rmtree(pkg, ignore_errors=True)
pkg.mkdir(parents=True, exist_ok=True)
# Copy only adapter_config.json into pkg to edit it; the large
# adapter_model.safetensors is zipped straight from OUTPUT_DIR (not duplicated).
shutil.copy2(OUTPUT_DIR / "adapter_config.json", pkg / "adapter_config.json")

cfg = json.loads((pkg / "adapter_config.json").read_text())
cfg["inference_mode"] = True
cfg["base_model_name_or_path"] = BASE_MODEL_HF

ref_zip = Path(REFERENCE_SUBMISSION_ZIP)
if ref_zip.exists():
    ref_dir = Path("/kaggle/working/ref")
    shutil.rmtree(ref_dir, ignore_errors=True)
    ref_dir.mkdir(parents=True)
    with zipfile.ZipFile(ref_zip) as z:
        z.extractall(ref_dir)
    ref_cfg_path = ref_dir / "adapter_config.json"
    if ref_cfg_path.exists():
        ref_cfg = json.loads(ref_cfg_path.read_text())
        for k in [
            "alora_invocation_tokens", "arrow_config", "ensure_weight_tying",
            "peft_version", "qalora_group_size", "target_parameters", "use_qalora",
        ]:
            if k in ref_cfg:
                cfg[k] = ref_cfg[k]
(pkg / "adapter_config.json").write_text(json.dumps(cfg, indent=2))

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pkg / "adapter_config.json", "adapter_config.json")
    z.write(OUTPUT_DIR / "adapter_model.safetensors", "adapter_model.safetensors")
print("built", SUBMISSION_ZIP, flush=True)

# ---------------------------------------------------------------------------
# Final task accuracy AFTER training (submission is already safe above).
# ---------------------------------------------------------------------------
if MAX_STEPS in PROBE_AT_STEPS:
    metrics.probes.append(task_accuracy("final", MAX_STEPS))

# Persist the full loss/eval/probe history for post-hoc inspection.
METRICS_JSON.write_text(
    json.dumps({"history": metrics.history, "probes": metrics.probes}, indent=2),
    encoding="utf-8",
)
print("wrote", METRICS_JSON)

# Headline: baseline -> final task accuracy per bucket (did it actually learn?).
_base = next((p for p in metrics.probes if p["tag"] == "baseline"), None)
_fin = next((p for p in metrics.probes if p["tag"] == "final"), None)
if _base and _fin:
    print("\n=== task accuracy: baseline -> final ===")
    for b in sorted(set(_base["per_bucket"]) | set(_fin["per_bucket"])):
        bc, bt = _base["per_bucket"].get(b, (0, 0))
        fc, ft = _fin["per_bucket"].get(b, (0, 0))
        ba = bc / bt if bt else 0.0
        fa = fc / ft if ft else 0.0
        print(f"  {b:<8} {ba:.2f} -> {fa:.2f}  ({'+' if fa >= ba else ''}{fa-ba:.2f})")
    print(f"  overall  {_base['acc']:.3f} -> {_fin['acc']:.3f}")


In [ ]:
# Render the run to PNGs: held-out eval-loss curve + baseline->final task
# accuracy per bucket. Non-fatal so a plotting error can't affect submission.zip.
try:
    import json
    from pathlib import Path

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    M = json.loads(Path("/kaggle/working/training_metrics.json").read_text())
    hist, probes = M.get("history", []), M.get("probes", [])

    tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
    ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

    if tr:
        ax[0].plot(*zip(*tr), color="#bbb", lw=1, label="train (batch=1, noisy)")
    if ev:
        ax[0].plot(*zip(*ev), color="#1f77b4", marker="o", lw=2, label="eval (held-out)")
    ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].set_title("Loss")
    ax[0].legend(); ax[0].grid(alpha=0.3)

    base = next((p for p in probes if p["tag"] == "baseline"), None)
    fin = next((p for p in probes if p["tag"] == "final"), None)
    if base and fin:
        buckets = sorted(set(base["per_bucket"]) | set(fin["per_bucket"]))

        def _acc(p, b):
            c, t = p["per_bucket"].get(b, (0, 0))
            return c / t if t else 0.0

        ba = [_acc(base, b) for b in buckets]
        fa = [_acc(fin, b) for b in buckets]
        x = range(len(buckets)); w = 0.38
        ax[1].bar([i - w / 2 for i in x], ba, w, label=f"baseline ({base['acc']:.2f})", color="#aaa")
        ax[1].bar([i + w / 2 for i in x], fa, w, label=f"final ({fin['acc']:.2f})", color="#2ca02c")
        ax[1].set_xticks(list(x)); ax[1].set_xticklabels(buckets, rotation=20)
        ax[1].set_ylim(0, 1); ax[1].set_ylabel("answer accuracy")
        ax[1].set_title("Task accuracy: baseline -> final")
        ax[1].legend(); ax[1].grid(axis="y", alpha=0.3)
    else:
        ax[1].text(0.5, 0.5, "no probe data", ha="center", va="center")

    fig.tight_layout()
    out = "/kaggle/working/training_report.png"
    fig.savefig(out, dpi=130, bbox_inches="tight")
    print("saved", out, flush=True)
    try:
        from IPython.display import Image, display
        display(Image(filename=out))
    except Exception:
        pass
except Exception as e:
    print("viz skipped:", e, flush=True)
